Пытаюсь достать данные с апи

In [1]:
import requests
import json
import pandas as pd
from config import APP_SECRET_KEY, USER_AGENT, BASE_URL

def get_headers():
    headers = {
        'X-Api-App-Id': APP_SECRET_KEY,
        'User-Agent': USER_AGENT
    }
    return headers


def get_towns(keyword=''):
    
    headers = get_headers()
    request = f'{BASE_URL}/towns/?keyword={keyword}'

    return get_res_by_request_and_headers(request, headers)


def get_catalogues():
    
    headers = get_headers()
    request = f'{BASE_URL}/catalogues/'
    
    return get_res_by_request_and_headers(request, headers)


def get_vacancies_page(page_id=0, count=100, keyword='', town=None, catalogue=None):
    
    headers = get_headers()
    request = f'{BASE_URL}/vacancies/?page={page_id}&count={count}&keyword={keyword}'

    if town is not None:
        request += f'&town={town}'

    if catalogue is not None:
        request += f'&catalogues={catalogue}'
        
    return get_res_by_request_and_headers(request, headers)


def get_vacancy(vacancy_id):
    
    headers = get_headers()
    request = f'{BASE_URL}/vacancies/{vacancy_id}/'
    
    return get_res_by_request_and_headers(request, headers)



def get_res_by_request_and_headers(request, headers):
    return requests.get(request, headers=headers).json()




In [9]:
import pandas as pd

In [30]:
def get_company(company_id):

    headers = get_headers()
    request = f'{BASE_URL}/clients/{company_id}/'
    
    return get_res_by_request_and_headers(request, headers)

In [36]:
def vacancies_to_df(vacancies):
    rows = []
    companies_info = {}

    for v in vacancies:
        company_id = v.get('id_client')
        if company_id:
            if company_id not in companies_info:
                companies_info[company_id] = get_company(company_id)

        row = {
            'id': v.get('id'),
            'Название вакансии': v.get('profession'),
            'Минимальная зарплата': v.get('payment_from'),
            'Максимальная зарплата': v.get('payment_to'),
            'Валюта': v.get('currency'),
            'Город': v.get('town', {}).get('title'),
            'Метро': v.get('metro'),
            'Адрес': v.get('address'),
            'Id компании': v.get('id_client'),
            'Индутрия': companies_info[company_id].get('industry')[0].get('title') if companies_info[company_id].get('industry') else None,
            'Описание компании': companies_info[company_id].get('description') if companies_info[company_id].get('description') else None,
            'Название компании': v.get('client', {}).get('title'),
            'Опыт': v.get('experience', {}).get('title'),
            'График работы': v.get('type_of_work',{}).get('title'),
            'Образование': v.get('education',{}).get('title'),
            'Категории': v.get('catalogues')[0].get('title') if v.get('catalogues') else None,
            'Широта': v.get('latitude'),
            'Долгота': v.get('longitude'),
            'Описание': v.get('candidat'),
            'ссылка': v.get('link')
        }
        rows.append(row)

    return pd.DataFrame(rows)

In [37]:
vacancies = [get_vacancy(51698248)]
df = vacancies_to_df(vacancies)

df.head()

,id,Название вакансии,Минимальная зарплата,Максимальная зарплата,Валюта,Город,Метро,Адрес,Id компании,Индутрия,Описание компании,Название компании,Опыт,График работы,Образование,Категории,Широта,Долгота,Описание,ссылка
0,51698248,Senior Python-разработчик,0,575000,rub,Москва,[],None,4942874,None,Точка – полностью онлайновый банковский сервис...,Точка банк,От 3 лет,Полный рабочий день,Не имеет значения,"IT, Интернет, связь, телеком",None,None,Ищем опытного Python-разработчика в Точка Банк...,https://www.superjob.ru/vakansii/senior-python...


In [40]:
df['Описание компании']

0    Точка – полностью онлайновый банковский сервис...
Name: Описание компании, dtype: object

In [62]:
towns_data = get_towns('Екатеринбург')


In [63]:
towns_data

{'objects': [{'id': 33,
   'id_region': 65,
   'id_country': 1,
   'title': 'Екатеринбург',
   'title_eng': 'Ekaterinburg'}],
 'total': 1,
 'more': False}

In [26]:
towns_data.get('objects')[0].get('id')

14

In [27]:
catalogues_data = get_catalogues()

In [43]:
catalogues_data

[{'title_rus': 'IT, Интернет, связь, телеком',
  'url_rus': 'it-internet-svyaz-telekom',
  'title': 'IT, Интернет, связь, телеком',
  'title_trimmed': 'IT, Интернет, связь,...',
  'key': 33,
  'positions': [{'title_rus': 'AI',
    'url_rus': 'ai',
    'title': 'AI',
    'id_parent': 33,
    'key': 651},
   {'title_rus': 'CRM-системы',
    'url_rus': 'crm-sistemy',
    'title': 'CRM-системы',
    'id_parent': 33,
    'key': 603},
   {'title_rus': 'Data Science',
    'url_rus': 'data-science',
    'title': 'Data Science',
    'id_parent': 33,
    'key': 627},
   {'title_rus': 'DevOps',
    'url_rus': 'devops',
    'title': 'DevOps',
    'id_parent': 33,
    'key': 628},
   {'title_rus': 'SRE',
    'url_rus': 'sre',
    'title': 'SRE',
    'id_parent': 33,
    'key': 629},
   {'title_rus': 'Web-верстка',
    'url_rus': 'web-verstka',
    'title': 'Web-верстка',
    'id_parent': 33,
    'key': 36},
   {'title_rus': 'Администрирование баз данных',
    'url_rus': 'administrirovanie-baz-danny

In [45]:
for cat in catalogues_data:
    if 'IT' in cat.get('title_rus'):
        id = cat.get('key')
id

33

In [70]:
def get_metro(town_id):

    headers = get_headers()
    request = f'{BASE_URL}/metro/{town_id}/lines/'
    
    return get_res_by_request_and_headers(request, headers)

In [71]:
data_metro = get_metro(4)

In [81]:
data_metro

[{'id': 1,
  'title': 'Сокольническая',
  'color': 'red',
  'stations': [{'id': 11, 'title': 'Библиотека им. Ленина'},
   {'id': 1, 'title': 'Бульвар Рокоссовского'},
   {'id': 16, 'title': 'Воробьёвы горы'},
   {'id': 6, 'title': 'Комсомольская'},
   {'id': 5, 'title': 'Красносельская'},
   {'id': 7, 'title': 'Красные ворота'},
   {'id': 12, 'title': 'Кропоткинская'},
   {'id': 9, 'title': 'Лубянка'},
   {'id': 629, 'title': 'Новомосковская'},
   {'id': 628, 'title': 'Ольховая'},
   {'id': 10, 'title': 'Охотный ряд'},
   {'id': 13, 'title': 'Парк Культуры'},
   {'id': 772, 'title': 'Потапово'},
   {'id': 3, 'title': 'Преображенская площадь'},
   {'id': 627, 'title': 'Прокшино'},
   {'id': 18, 'title': 'Проспект Вернадского'},
   {'id': 572, 'title': 'Румянцево'},
   {'id': 574, 'title': 'Саларьево'},
   {'id': 4, 'title': 'Сокольники'},
   {'id': 15, 'title': 'Спортивная'},
   {'id': 570, 'title': 'Тропарёво'},
   {'id': 17, 'title': 'Университет'},
   {'id': 626, 'title': 'Филатов Лу

In [ ]:
a = 769
line_id = 1

for line in data_metro:
    for item in line.get('stations'):
        if a == item.get('id'):
            final_line = line_id
            break
    line_id += 1 


data_metro[final_line-1].get('title'), data_metro[final_line-1].get('color')

('Троицкая', 'darkspringgreen')

In [2]:
def get_regions(keyword=''):

    headers = get_headers()
    request = f'{BASE_URL}/regions/?keyword={keyword}'

    return get_res_by_request_and_headers(request, headers)

In [10]:
data = get_regions('московская область')

In [11]:
data

{'objects': [{'id': 46, 'id_country': 1, 'title': 'Московская область'}],
 'total': 1,
 'more': False}

In [18]:
id = data.get('objects')[0].get('id')

In [19]:
id

46

In [20]:
data = get_vacancy(51820279)

In [21]:
data

{'canEdit': False,
 'is_closed': False,
 'id': 51820279,
 'id_client': 2635306,
 'payment_from': 83200,
 'payment_to': 106400,
 'date_pub_to': 1776647737,
 'date_archived': 0,
 'date_published': 1775697300,
 'address': None,
 'profession': 'Пекарь (Санкт-Петербург, Пулковское (поселок Шушары), Образцовая, 2)',
 'work': None,
 'compensation': None,
 'candidat': '«Пятерочка» приглашает на вакансию: Пекарь\n \nОТ НАС:\n• Быстрое оформление по ТК РФ - с помощью Госуслуг\n• Фиксированный оклад + премии и надбавки за стаж. Средний доход 83200 – 106400 руб. в месяц до вычета налогов\n• График работы 5/2, 2/2, возможен неполный рабочий день, неполная рабочая неделя\n• Возможность профессионального развития и быстрый карьерный рост до директора магазина\n• Финансовая поддержка в сложных жизненных ситуациях\n• Подписка на сервис «Пакет» с повышенным кэшбеком только для своих сотрудников, скидки и акции от наших партнёров\n• Медицинская книжка за счет компании\n• Оборудованная комната отдыха с бе